# Demo 3 — How do we select once and evaluate honestly?

**Learning question:** Does a train-only linear Pipeline beat a training-mean baseline on validation MAE, and what does one later held-out period show after that choice is frozen?

The input grain is one synthetic Station A daily prediction issue. Output grains are one metric row per validation approach, one final test-metric row, one row per held-out prediction, one supplied binary-prediction metric row per approach, and one residual point per held-out issue. This Colab-first notebook runs equivalently in local Jupyter. Colab files are ephemeral, and edits opened from GitHub are not automatically saved back to GitHub. Assignment use of Colab remains conditional on the repository-save and Classroom 50 pilot. Use only the synthetic, non-identifying fixture; do not add private data or credentials. Restart the kernel and run every cell in order because stored output is not fresh-execution evidence.

In [ ]:
from importlib import metadata
from pathlib import Path
import platform
import subprocess
import sys

EXPECTED_PYTHON = "3.12.13"
EXPECTED_DISTRIBUTIONS = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "statsmodels": "0.14.6",
    "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1",
}

def distribution_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None

mismatched = [
    f"{name}=={expected}"
    for name, expected in EXPECTED_DISTRIBUTIONS.items()
    if distribution_version(name) != expected
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

actual_versions = {
    name: distribution_version(name) for name in EXPECTED_DISTRIBUTIONS
}
assert platform.python_version() == EXPECTED_PYTHON
assert actual_versions == EXPECTED_DISTRIBUTIONS

starting_directory = Path.cwd().resolve()
search_bases = (starting_directory, *starting_directory.parents)
demo_root = None
for search_base in search_bases:
    for candidate in (search_base, search_base / "10" / "demo"):
        if (candidate / "DEMO_GUIDE.md").is_file() and (
            candidate / ".python-version"
        ).is_file():
            demo_root = candidate
            break
    if demo_root is not None:
        break
DEMO_ROOT = demo_root if demo_root is not None else starting_directory

print(f"Python {platform.python_version()}")
for distribution_name, distribution_version_text in actual_versions.items():
    print(f"{distribution_name} {distribution_version_text}")
print(f"Demo root: {DEMO_ROOT}")

## Rebuild the prediction contract and partitions independently

A **training partition** estimates model state. A **validation partition** compares development choices with a predeclared metric. A **test partition** is an untouched final check after those choices are frozen. Here each unit is one Station A daily issue at 00:00 UTC, the target is next-day temperature, the horizon is one day, the feature cutoff is the issue time, and the primary validation metric is MAE.

The fixed target-time cutoffs create 22 training, 7 validation, and 11 test rows. This notebook derives those roles from the fixture and never reads Demo 2 output.

In [ ]:
import hashlib
import io
import numpy as np
import pandas as pd

STATION_BYTES = b'row_id,prediction_timestamp,target_timestamp,day_number,current_temperature_c,previous_temperature_c,target_next_day_temperature_c\nstation-a-20260102,2026-01-02T00:00:00Z,2026-01-03T00:00:00Z,1,10.752852,10.400000,11.150020\nstation-a-20260103,2026-01-03T00:00:00Z,2026-01-04T00:00:00Z,2,11.150020,10.752852,12.284133\nstation-a-20260104,2026-01-04T00:00:00Z,2026-01-05T00:00:00Z,3,12.284133,11.150020,12.891635\nstation-a-20260105,2026-01-05T00:00:00Z,2026-01-06T00:00:00Z,4,12.891635,12.284133,12.500011\nstation-a-20260106,2026-01-06T00:00:00Z,2026-01-07T00:00:00Z,5,12.500011,12.891635,12.432889\nstation-a-20260107,2026-01-07T00:00:00Z,2026-01-08T00:00:00Z,6,12.432889,12.500011,12.810600\nstation-a-20260108,2026-01-08T00:00:00Z,2026-01-09T00:00:00Z,7,12.810600,12.432889,12.319227\nstation-a-20260109,2026-01-09T00:00:00Z,2026-01-10T00:00:00Z,8,12.319227,12.810600,11.265068\nstation-a-20260110,2026-01-10T00:00:00Z,2026-01-11T00:00:00Z,9,11.265068,12.319227,11.008799\nstation-a-20260111,2026-01-11T00:00:00Z,2026-01-12T00:00:00Z,10,11.008799,11.265068,11.042981\nstation-a-20260112,2026-01-12T00:00:00Z,2026-01-13T00:00:00Z,11,11.042981,11.008799,10.294535\nstation-a-20260113,2026-01-13T00:00:00Z,2026-01-14T00:00:00Z,12,10.294535,11.042981,9.694338\nstation-a-20260114,2026-01-14T00:00:00Z,2026-01-15T00:00:00Z,13,9.694338,10.294535,10.196415\nstation-a-20260115,2026-01-15T00:00:00Z,2026-01-16T00:00:00Z,14,10.196415,9.694338,10.705477\nstation-a-20260116,2026-01-16T00:00:00Z,2026-01-17T00:00:00Z,15,10.705477,10.196415,10.582814\nstation-a-20260117,2026-01-17T00:00:00Z,2026-01-18T00:00:00Z,16,10.582814,10.705477,11.069374\nstation-a-20260118,2026-01-18T00:00:00Z,2026-01-19T00:00:00Z,17,11.069374,10.582814,12.415247\nstation-a-20260119,2026-01-19T00:00:00Z,2026-01-20T00:00:00Z,18,12.415247,11.069374,13.203857\nstation-a-20260120,2026-01-20T00:00:00Z,2026-01-21T00:00:00Z,19,13.203857,12.415247,13.408874\nstation-a-20260121,2026-01-21T00:00:00Z,2026-01-22T00:00:00Z,20,13.408874,13.203857,14.297838\nstation-a-20260122,2026-01-22T00:00:00Z,2026-01-23T00:00:00Z,21,14.297838,13.408874,15.417233\nstation-a-20260123,2026-01-23T00:00:00Z,2026-01-24T00:00:00Z,22,15.417233,14.297838,15.482652\nstation-a-20260124,2026-01-24T00:00:00Z,2026-01-25T00:00:00Z,23,15.482652,15.417233,15.179048\nstation-a-20260125,2026-01-25T00:00:00Z,2026-01-26T00:00:00Z,24,15.179048,15.482652,15.559942\nstation-a-20260126,2026-01-26T00:00:00Z,2026-01-27T00:00:00Z,25,15.559942,15.179048,15.665661\nstation-a-20260127,2026-01-27T00:00:00Z,2026-01-28T00:00:00Z,26,15.665661,15.559942,14.738241\nstation-a-20260128,2026-01-28T00:00:00Z,2026-01-29T00:00:00Z,27,14.738241,15.665661,14.027121\nstation-a-20260129,2026-01-29T00:00:00Z,2026-01-30T00:00:00Z,28,14.027121,14.738241,14.098535\nstation-a-20260130,2026-01-30T00:00:00Z,2026-01-31T00:00:00Z,29,14.098535,14.027121,13.708819\nstation-a-20260131,2026-01-31T00:00:00Z,2026-02-01T00:00:00Z,30,13.708819,14.098535,12.768661\nstation-a-20260201,2026-02-01T00:00:00Z,2026-02-02T00:00:00Z,31,12.768661,13.708819,12.688712\nstation-a-20260202,2026-02-02T00:00:00Z,2026-02-03T00:00:00Z,32,12.688712,12.768661,13.310430\nstation-a-20260203,2026-02-03T00:00:00Z,2026-02-04T00:00:00Z,33,13.310430,12.688712,13.338624\nstation-a-20260204,2026-02-04T00:00:00Z,2026-02-05T00:00:00Z,34,13.338624,13.310430,13.290932\nstation-a-20260205,2026-02-05T00:00:00Z,2026-02-06T00:00:00Z,35,13.290932,13.338624,14.302447\nstation-a-20260206,2026-02-06T00:00:00Z,2026-02-07T00:00:00Z,36,14.302447,13.290932,15.487204\nstation-a-20260207,2026-02-07T00:00:00Z,2026-02-08T00:00:00Z,37,15.487204,14.302447,15.821827\nstation-a-20260208,2026-02-08T00:00:00Z,2026-02-09T00:00:00Z,38,15.821827,15.487204,16.311473\nstation-a-20260209,2026-02-09T00:00:00Z,2026-02-10T00:00:00Z,39,16.311473,15.821827,17.563960\nstation-a-20260210,2026-02-10T00:00:00Z,2026-02-11T00:00:00Z,40,17.563960,16.311473,18.266177\n'
STATION_SHA256 = "f95330b252c6e0f12026577602c69e21d01dbec232b5e523c6c41b0b62cf85a8"
TIMESTAMP_FORMAT = "%Y-%m-%dT%H:%M:%SZ"
fixture_path = DEMO_ROOT / "data" / "station_next_day.csv"
fixture_bytes = fixture_path.read_bytes() if fixture_path.is_file() else STATION_BYTES
assert len(fixture_bytes) == 3879
assert hashlib.sha256(fixture_bytes).hexdigest() == STATION_SHA256

station_data = pd.read_csv(
    io.BytesIO(fixture_bytes),
    dtype={
        "row_id": "string",
        "prediction_timestamp": "string",
        "target_timestamp": "string",
        "day_number": "int64",
        "current_temperature_c": "float64",
        "previous_temperature_c": "float64",
        "target_next_day_temperature_c": "float64",
    },
)
for timestamp_column in ["prediction_timestamp", "target_timestamp"]:
    station_data[timestamp_column] = pd.to_datetime(
        station_data[timestamp_column], format=TIMESTAMP_FORMAT, utc=True
    ).astype("datetime64[us, UTC]")
assert station_data.shape == (40, 7)
assert station_data["row_id"].is_unique
assert station_data.notna().all().all()
assert [str(dtype) for dtype in station_data.dtypes] == [
    "string", "datetime64[us, UTC]", "datetime64[us, UTC]",
    "int64", "float64", "float64", "float64"
]
assert station_data["prediction_timestamp"].is_monotonic_increasing
assert (
    station_data["target_timestamp"] - station_data["prediction_timestamp"]
    == pd.Timedelta(days=1)
).all()

validation_start = pd.Timestamp("2026-01-25T00:00:00Z").as_unit("us")
test_start = pd.Timestamp("2026-02-01T00:00:00Z").as_unit("us")
train_frame = station_data.loc[station_data["target_timestamp"] < validation_start].copy()
validation_frame = station_data.loc[
    (station_data["target_timestamp"] >= validation_start)
    & (station_data["target_timestamp"] < test_start)
].copy()
test_frame = station_data.loc[station_data["target_timestamp"] >= test_start].copy()
assert [len(train_frame), len(validation_frame), len(test_frame)] == [22, 7, 11]
assert train_frame["target_timestamp"].max() < validation_frame["target_timestamp"].min()
assert validation_frame["target_timestamp"].max() < test_frame["target_timestamp"].min()
partition_ids = [
    set(train_frame["row_id"]),
    set(validation_frame["row_id"]),
    set(test_frame["row_id"]),
]
assert set.union(*partition_ids) == set(station_data["row_id"])
assert all(
    partition_ids[left].isdisjoint(partition_ids[right])
    for left in range(3) for right in range(left + 1, 3)
)

OUTPUT_DIR = DEMO_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
demo3_output_names = [
    "validation_metrics.csv",
    "final_test_metrics.csv",
    "final_predictions.csv",
    "prediction_residuals.png",
    "binary_metrics.csv",
]
for output_name in demo3_output_names:
    owned_path = OUTPUT_DIR / output_name
    if owned_path.exists():
        owned_path.unlink()

display(pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "rows": [len(train_frame), len(validation_frame), len(test_frame)],
}))

## Define the two approaches before fitting

An **estimator** learns a rule from data. **Fit** estimates its state from training rows; **predict** applies frozen state to feature rows. **Preprocessing** transforms model inputs. `StandardScaler` learns training-column means and scales. A `Pipeline` connects preprocessing and an estimator so the same training-fitted transformation is applied later. A **baseline** is a simple reference; ours predicts the training-target mean.

Fit exactly two regression approaches on training rows: the mean baseline and one `Pipeline(StandardScaler, LinearRegression)`. Features are `day_number`, `current_temperature_c`, and `previous_temperature_c`; the target is `target_next_day_temperature_c`. Do not preprocess before splitting or refit on validation/test.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_columns = [
    "day_number",
    "current_temperature_c",
    "previous_temperature_c",
]
target_column = "target_next_day_temperature_c"

baseline = DummyRegressor(strategy="mean")
linear_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]
)
baseline.fit(train_frame[feature_columns], train_frame[target_column])
linear_pipeline.fit(train_frame[feature_columns], train_frame[target_column])
fitted_approaches = {
    "training_mean_baseline": baseline,
    "linear_pipeline": linear_pipeline,
}

assert np.isclose(baseline.constant_.item(), 12.112455318181818)
expected_scaler_means = np.array([
    11.5,
    11.897464409090908,
    11.669408363636364,
])
fitted_scaler_means = linear_pipeline.named_steps["scale"].mean_
assert np.allclose(fitted_scaler_means, expected_scaler_means)
assert np.allclose(
    fitted_scaler_means,
    train_frame[feature_columns].mean().to_numpy(),
)
assert not np.allclose(
    fitted_scaler_means,
    station_data[feature_columns].mean().to_numpy(),
)

prediction_call_ledger = []
def record_predictions(approach_name, estimator, feature_frame, row_ids, partition):
    predictions = estimator.predict(feature_frame)
    prediction_call_ledger.append({
        "approach": approach_name,
        "partition": partition,
        "row_ids": tuple(row_ids.tolist()),
        "row_count": len(feature_frame),
    })
    return predictions

## Predeclare model selection on validation MAE

**Mean absolute error (MAE)** averages `abs(actual - predicted)` in target units; lower is better. **Root mean squared error (RMSE)** is the square root of mean squared error and gives larger errors more weight; lower is better. **R2** compares squared error with predicting the partition's actual-value mean; 1 is perfect, 0 matches that reference, and negative values are worse.

MAE is the sole selection metric. Before running validation, predict which approach will have lower MAE. RMSE and R2 describe the same validation predictions but do not change the rule.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    root_mean_squared_error,
)

validation_rows = []
for approach_name, fitted_estimator in fitted_approaches.items():
    validation_predictions = record_predictions(
        approach_name,
        fitted_estimator,
        validation_frame[feature_columns],
        validation_frame["row_id"],
        "validation",
    )
    validation_rows.append({
        "approach": approach_name,
        "mae": mean_absolute_error(validation_frame[target_column], validation_predictions),
        "rmse": root_mean_squared_error(validation_frame[target_column], validation_predictions),
        "r2": r2_score(validation_frame[target_column], validation_predictions),
    })
validation_metrics = pd.DataFrame(validation_rows).astype({
    "approach": "string", "mae": "float64", "rmse": "float64", "r2": "float64"
})
assert validation_metrics["approach"].tolist() == [
    "training_mean_baseline", "linear_pipeline"
]
assert np.allclose(
    validation_metrics[["mae", "rmse", "r2"]].to_numpy(),
    [[2.598597, 2.698360, -12.778632], [0.408166, 0.494871, 0.536563]],
    atol=5e-7,
)
selected_approach_name = validation_metrics.sort_values(
    ["mae", "approach"], kind="stable"
).iloc[0]["approach"]
assert selected_approach_name == "linear_pipeline"
assert [entry["partition"] for entry in prediction_call_ledger] == [
    "validation", "validation"
]

validation_path = OUTPUT_DIR / "validation_metrics.csv"
validation_metrics.to_csv(
    validation_path,
    index=False,
    lineterminator="\n",
    float_format="%.6f",
)
assert len(validation_path.read_bytes()) == 116
assert hashlib.sha256(validation_path.read_bytes()).hexdigest() == (
    "899d246c84c5116e857e81a4327055819d3bef1d214e4199e6247f82fd69d25f"
)
display(validation_metrics)
print(f"Frozen selection from validation MAE: {selected_approach_name}")

## Freeze the choice before final evaluation

**Final evaluation** applies the already-selected, training-fitted approach to the untouched test partition once. It describes performance on this held-out period; it is not another development round. The selected name, features, preprocessing, estimator state, and metrics are now frozen.

In the residual plot, `actual - predicted` above zero means the day was warmer than predicted; below zero means it was cooler. Predict the signs before revealing the points. If this test result prompts redevelopment, it becomes development evidence and a new untouched test release is required.

In [ ]:
selected_estimator = fitted_approaches[selected_approach_name]
final_prediction_values = record_predictions(
    selected_approach_name,
    selected_estimator,
    test_frame[feature_columns],
    test_frame["row_id"],
    "test",
)
final_test_metrics = pd.DataFrame({
    "approach": pd.Series([selected_approach_name], dtype="string"),
    "partition": pd.Series(["test"], dtype="string"),
    "mae": pd.Series([
        mean_absolute_error(test_frame[target_column], final_prediction_values)
    ], dtype="float64"),
    "rmse": pd.Series([
        root_mean_squared_error(test_frame[target_column], final_prediction_values)
    ], dtype="float64"),
    "r2": pd.Series([
        r2_score(test_frame[target_column], final_prediction_values)
    ], dtype="float64"),
})
final_predictions = pd.DataFrame({
    "row_id": test_frame["row_id"].reset_index(drop=True).astype("string"),
    "target_timestamp": test_frame["target_timestamp"].reset_index(drop=True),
    "actual_temperature_c": test_frame[target_column].reset_index(drop=True).astype("float64"),
    "predicted_temperature_c": pd.Series(final_prediction_values, dtype="float64"),
})
final_predictions["residual_c"] = (
    final_predictions["actual_temperature_c"]
    - final_predictions["predicted_temperature_c"]
).astype("float64")
assert final_test_metrics["approach"].item() == "linear_pipeline"
assert np.allclose(
    final_test_metrics[["mae", "rmse", "r2"]].iloc[0].to_numpy(dtype="float64"),
    [0.450242, 0.542081, 0.916920],
    atol=5e-7,
)
assert len(final_predictions) == 11
assert np.allclose(
    final_predictions["predicted_temperature_c"].to_numpy(),
    [13.742458, 12.652152, 13.076531, 14.019072, 13.740149, 13.679424,
     15.153987, 16.292481, 16.137180, 16.666313, 18.186271],
    atol=5e-7,
)
assert len([entry for entry in prediction_call_ledger if entry["partition"] == "test"]) == 1
assert prediction_call_ledger[-1]["row_ids"] == tuple(test_frame["row_id"].tolist())
assert prediction_call_ledger[-1]["row_count"] == 11

final_metrics_path = OUTPUT_DIR / "final_test_metrics.csv"
final_predictions_path = OUTPUT_DIR / "final_predictions.csv"
final_test_metrics.to_csv(
    final_metrics_path, index=False, lineterminator="\n", float_format="%.6f"
)
final_predictions.to_csv(
    final_predictions_path,
    index=False,
    lineterminator="\n",
    float_format="%.6f",
    date_format=TIMESTAMP_FORMAT,
)
assert len(final_metrics_path.read_bytes()) == 79
assert hashlib.sha256(final_metrics_path.read_bytes()).hexdigest() == "8c4163c7d9f5de0a49f41646d30a68b4569e96c1e7b69d0d89f7706fbc3ffc6a"
assert len(final_predictions_path.read_bytes()) == 843
assert hashlib.sha256(final_predictions_path.read_bytes()).hexdigest() == "eb2b5a5ffa2d62cdfb6f92b1a6e7fea0c2c5c4b1d33e433f4d2cd5c568c08090"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

prediction_figure, prediction_axis = plt.subplots(figsize=(10, 6))
prediction_axis.scatter(
    final_predictions["predicted_temperature_c"],
    final_predictions["residual_c"],
    color="#1f4e79",
    edgecolor="white",
    linewidth=0.7,
    s=70,
    zorder=3,
)
prediction_axis.axhline(0.0, color="#b22222", linewidth=1.8, zorder=2)
prediction_axis.set_title("Final test residuals: selected linear Pipeline")
prediction_axis.set_xlabel("Predicted next-day temperature (degrees C)")
prediction_axis.set_ylabel("Residual (actual - predicted, degrees C)")
prediction_axis.grid(alpha=0.2, zorder=1)
assert len(prediction_axis.collections[0].get_offsets()) == 11
assert np.allclose(prediction_axis.lines[0].get_ydata(), [0.0, 0.0])
prediction_figure.tight_layout()
prediction_plot_path = OUTPUT_DIR / "prediction_residuals.png"
prediction_figure.savefig(prediction_plot_path, dpi=120)
display(final_test_metrics)
display(final_predictions)
display(prediction_figure)
plt.close(prediction_figure)
print("This residual plot describes only the supplied held-out period.")

## Calculate supplied binary metrics without fitting a classifier

**Binary classification** assigns one of two classes. The **positive class** is the event labeled 1. A **true positive** predicts 1 when actual is 1; a **false positive** predicts 1 when actual is 0; a **false negative** predicts 0 when actual is 1. **Accuracy** is the fraction of all labels predicted correctly. **Precision** is true positives divided by predicted positives. **Recall** is true positives divided by actual positives.

Before calculating, predict why the supplied model and all-zero supplied dummy can have equal accuracy but different recall. The relevant tradeoff depends on decision consequences; neither supplied prediction column is globally better.

Limit the conclusions: data are synthetic; they cover one station and date range, fixed features, conditional model assumptions, possible distribution change, and one held-out period. Results support no causal claim or universal performance guarantee.

In [ ]:
actual_binary = np.array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype="int64")
supplied_predictions = {
    "supplied_model": np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0], dtype="int64"),
    "supplied_dummy": np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype="int64"),
}
binary_rows = []
for approach_name, prediction_labels in supplied_predictions.items():
    binary_rows.append({
        "approach": approach_name,
        "accuracy": accuracy_score(actual_binary, prediction_labels),
        "precision": precision_score(
            actual_binary, prediction_labels, pos_label=1, zero_division=0
        ),
        "recall": recall_score(
            actual_binary, prediction_labels, pos_label=1, zero_division=0
        ),
    })
binary_metrics = pd.DataFrame(binary_rows).astype({
    "approach": "string",
    "accuracy": "float64",
    "precision": "float64",
    "recall": "float64",
})
assert np.allclose(
    binary_metrics[["accuracy", "precision", "recall"]].to_numpy(),
    [[0.8, 0.5, 0.5], [0.8, 0.0, 0.0]],
)
binary_path = OUTPUT_DIR / "binary_metrics.csv"
binary_metrics.to_csv(
    binary_path,
    index=False,
    lineterminator="\n",
    float_format="%.6f",
)
assert len(binary_path.read_bytes()) == 119
assert hashlib.sha256(binary_path.read_bytes()).hexdigest() == (
    "52180318d2393b626ceeafc07a93ba751bcd7bb2f56907a6a9e746b783632ba3"
)
display(binary_metrics)
print(
    "Equal accuracy does not imply equal positive-class behavior: "
    "the supplied recalls are 0.5 and 0.0."
)

In [ ]:
import struct

validation_readback = pd.read_csv(
    validation_path,
    dtype={"approach": "string", "mae": "float64", "rmse": "float64", "r2": "float64"},
)
final_metrics_readback = pd.read_csv(
    final_metrics_path,
    dtype={"approach": "string", "partition": "string", "mae": "float64", "rmse": "float64", "r2": "float64"},
)
final_predictions_readback = pd.read_csv(
    final_predictions_path,
    dtype={
        "row_id": "string",
        "target_timestamp": "string",
        "actual_temperature_c": "float64",
        "predicted_temperature_c": "float64",
        "residual_c": "float64",
    },
)
final_predictions_readback["target_timestamp"] = pd.to_datetime(
    final_predictions_readback["target_timestamp"], format=TIMESTAMP_FORMAT, utc=True
).astype("datetime64[us, UTC]")
binary_readback = pd.read_csv(
    binary_path,
    dtype={"approach": "string", "accuracy": "float64", "precision": "float64", "recall": "float64"},
)
assert [str(dtype) for dtype in validation_readback.dtypes] == ["string", "float64", "float64", "float64"]
assert [str(dtype) for dtype in final_metrics_readback.dtypes] == ["string", "string", "float64", "float64", "float64"]
assert [str(dtype) for dtype in final_predictions_readback.dtypes] == [
    "string", "datetime64[us, UTC]", "float64", "float64", "float64"
]
assert [str(dtype) for dtype in binary_readback.dtypes] == ["string", "float64", "float64", "float64"]
assert validation_readback.equals(validation_metrics.round(6))
assert final_metrics_readback.equals(final_test_metrics.round(6))
expected_serialized_predictions = final_predictions.copy()
for numeric_column in ["actual_temperature_c", "predicted_temperature_c", "residual_c"]:
    expected_serialized_predictions[numeric_column] = expected_serialized_predictions[numeric_column].round(6)
assert final_predictions_readback.equals(expected_serialized_predictions)
assert binary_readback.equals(binary_metrics)

expected_hashes = {
    validation_path: (116, "899d246c84c5116e857e81a4327055819d3bef1d214e4199e6247f82fd69d25f"),
    final_metrics_path: (79, "8c4163c7d9f5de0a49f41646d30a68b4569e96c1e7b69d0d89f7706fbc3ffc6a"),
    final_predictions_path: (843, "eb2b5a5ffa2d62cdfb6f92b1a6e7fea0c2c5c4b1d33e433f4d2cd5c568c08090"),
    binary_path: (119, "52180318d2393b626ceeafc07a93ba751bcd7bb2f56907a6a9e746b783632ba3"),
}
for artifact_path, (expected_bytes, expected_hash) in expected_hashes.items():
    artifact_bytes = artifact_path.read_bytes()
    assert len(artifact_bytes) == expected_bytes
    assert hashlib.sha256(artifact_bytes).hexdigest() == expected_hash

png_bytes = prediction_plot_path.read_bytes()
assert png_bytes[:8] == b"\x89PNG\r\n\x1a\n"
assert struct.unpack(">II", png_bytes[16:24]) == (1200, 720)
assert png_bytes[25] in {2, 6}
assert selected_approach_name == "linear_pipeline"
assert [entry["partition"] for entry in prediction_call_ledger] == [
    "validation", "validation", "test"
]
assert prediction_call_ledger[-1]["row_ids"] == tuple(test_frame["row_id"].tolist())
assert set(path.name for path in OUTPUT_DIR.iterdir() if path.name in demo3_output_names) == set(demo3_output_names)
print("PASS: Demo 3 validation selection, one-use final evaluation, plots, and binary metrics verified.")